# GNN playground


## Small Signal Graph


### Graph Builder


In [ ]:
# graph_builder.py  ——  Parse netlist (Sky130) + Params from outside -> Graph (PyG Data or dict)
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any
import re
import pandas as pd

# ======== Schema enums ========
PIN_ROLE = {
    "G": 0,
    "D": 1,
    "S": 2,
    "B": 3,  # MOS pins
    "RT": 4,
    "RB": 5,  # Resistor pins
    "CT": 6,
    "CB": 7,  # Capacitor pins
    "IT": 8,
    "IB": 9,  # Current source
    "VDD": 10,
    "GND": 11,  # Supplies
    "dummy": 12,
    "NET": 13,
}
EDGE_TYPE = {"intra": 0, "inter": 1}
EDGE_ROLE = {
    "none": 0,
    "gm_control": 1,
    "gm_target": 2,
    "current_source": 3,
    "capacitor": 4,
}


# ======== small tool ========
def inst_key(raw: str) -> str:
    # "XM2" / "M2" / "m2" -> "M2"
    m = re.search(r"([xX]?[mM]\d+)", raw)
    return m.group(1).upper().replace("XM", "M") if m else raw.upper()


def is_pfet(model: str) -> bool:
    return "pfet" in model.lower()


def mos_type_id(model: str) -> int:
    """0 for nfet, 1 for pfet"""
    return 1 if is_pfet(model) else 0


def norm_name(n: str) -> str:
    return n.strip().lower()


# ======== analyze netlist lines (Sky130 MOS: D G S B)=======
@dataclass
class MosLine:
    name: str  # "M1"
    d: str
    g: str
    s: str
    b: str
    model: str  # "sky130_fd_pr__pfet_01v8"


@dataclass
class CapLine:
    name: str  # "C1"
    t: str
    b: str
    model: str  # "sky130_fd_pr__cap_mim_m3_1"


# TODO: support multi current source
@dataclass
class IbiasLine:
    name: str  # "I0"
    pos: str  # "vdda"
    neg: str  # "net3"


def parse_netlist_lines(
    text: str,
) -> Tuple[List[MosLine], List[CapLine], List[IbiasLine], Dict[str, str], List[str]]:
    mos_list: List[MosLine] = []
    cap_list: List[CapLine] = []
    ibias_list: List[IbiasLine] = []
    # supplies map, extendable
    supplies_map = {"vdda": "VDD", "gnda": "GND"}
    all_nets: set[str] = set()

    for raw in text.splitlines():
        s = raw.strip()
        # remove comments, subcircuit interface and empty lines
        if not s or s.startswith(("*", ";", "//", ".")):
            continue

        # MOS：  XM2 D G S B model
        if re.match(r"^[xX]?[mM]\d+\s", s):
            toks = s.split()
            name = inst_key(toks[0])  # XM2 -> M2
            d, g, s_, b = toks[1:5]  # Sky130 mosfet pin order: D G S B
            model = toks[5]

            mos_list.append(MosLine(name=name, d=d, g=g, s=s_, b=b, model=model))
            all_nets.update([d, g, s_, b])
            continue

        # CAP： XC1 nodeT nodeB model
        if re.match(r"^[xX]?[cC]\d+\s", s):
            toks = s.split()
            name = toks[0].upper().replace("XC", "C").replace("c", "C")
            t, b = toks[1], toks[2]
            model = toks[3]

            cap_list.append(CapLine(name=name, t=t, b=b, model=model))
            all_nets.update([t, b])
            continue

        # TODO: support multi current source, need to modify the regex
        # current source: I0 pos neg
        if re.match(r"^I0\s", s):
            toks = s.split()
            name = toks[0].upper()
            pos, neg = toks[1], toks[2]

            ibias_list.append(IbiasLine(name=name, pos=pos, neg=neg))
            all_nets.update([pos, neg])
            continue

        # other: only collect net names for later use
        toks = s.split()
        if len(toks) >= 3:
            all_nets.update(toks[1:3])

    # record all net names (for creating NET nodes)
    return mos_list, cap_list, ibias_list, supplies_map, sorted(all_nets)


# ========= op analysis ========
# op_csv_adapter.py  ——  mapping the row of DataFrame to the op_dict for building the graph

# only receive the small signal/bias names in the whitelist (prevent the introduction of cgg/cdd etc.)
OP_WHITELIST = {
    "gm",
    "gds",
    "gmbs",
    "vgs",
    "vds",
    "vbs",
    "vth",
    "vdsat",
    "id",
    "cgs",
    "cgd",
    "cdb",
    "csb",
}


def row_to_params_dict(row) -> Dict[str, Any]:
    """
    input: a row of DataFrame (pandas Series)
    output: the params_dict for building the graph, for example
    {'gm_m1':..., 'vgs_m2':..., 'capacitance_c1':..., 'M1_L':..., 'M1_WL_ratio':..., 'M1_M':..., 'C1_L':..., 'C1_W':...,}
    convention:
      - the column name is like 'gm_m1', 'cgd_m6', 'vds_m8' etc.
      - the capacitance_c1 is the capacitance of the capacitor C1 (if not exist, set to 0)
      - you can also put the global metrics in meta_*, for the graph-level attributes
    """
    params: Dict[str, Dict[str, Any]] = {}
    op: Dict[str, Any] = {}
    size: Dict[str, Any] = {}
    spec: Dict[str, Any] = {}

    for col, val in row.items():
        cname = str(col).strip().lower()

        # 1) MOS small signal / bias (whitelist + m\d suffix)
        m = re.match(rf"^({'|'.join(OP_WHITELIST)})_(m\d+)$", cname)
        if m:
            dev = m.group(2).upper()
            attr = m.group(1)
            op.setdefault(dev, {})[attr.lower()] = float(val)
            continue

        # 2) capacitance: capacitance_c1
        if re.match(r"^capacitance_c\d+$", cname):
            dev = cname.split("_")[1].upper()
            attr = cname.split("_")[0]
            op.setdefault(dev, {})[attr.lower()] = float(val)
            continue

        # TODO: Currently only support single current source, in the future, we need to support multi current source
        # 3) bias: ibias
        if re.match(r"^ibias_m\d+$", cname) or cname == "ibias":
            if cname == "ibias":
                dev = "ibias0"
                attr = "ibias"
            else:
                dev = cname.split("_")[1]
                attr = cname.split("_")[0]
            op.setdefault(dev.lower(), {})[attr.lower()] = float(val)
            continue

        # 4) Mos sizes
        m2 = re.match(r"^(m\d+)_(l|w|m|wl_ratio)$", cname)
        if m2:
            # M1, M3, ...
            dev = m2.group(1).upper()
            attr = m2.group(2)
            size.setdefault(dev, {})[attr.lower()] = float(val)
            continue

        # 5) Capacitor sizes
        m3 = re.match(r"^(c\d+)_(l|w|m)$", cname)
        if m3:
            # C1, ...
            dev = m3.group(1).upper()
            attr = m3.group(2)
            size.setdefault(dev, {})[attr.lower()] = float(val)
            continue

        # 6) global metrics (optional, for the graph-level attributes)
        if cname in {"dcgain_", "gain_bandwidth_product_", "phase_margin", "power"}:
            spec[f"meta::{cname}"] = float(val)

    params["op"] = op
    params["size"] = size
    params["spec"] = spec
    return params


# ======== read op and sizing data from csv file =======
def read_data_from_csv(
    csv_path: str, row_idx: Optional[int] = None
) -> List[Dict[str, Any]] or Dict[str, Any]:
    df = pd.read_csv(csv_path)
    all_params = []
    for _, row in df.iterrows():
        params = row_to_params_dict(row)
        all_params.append(params)
    return all_params if row_idx is None else all_params[row_idx]


# ======== build Graph ========
def build_graph_from_netlist_and_csv(
    netlist: str, csv_path: str, row_idx: int
) -> Dict[str, Any]:
    mos_list, cap_list, ibias_list, supplies_map, all_nets = parse_netlist_lines(
        netlist
    )
    all_params = read_data_from_csv(csv_path, row_idx)
    ops = all_params["op"]
    sizes = all_params["size"]
    specs = all_params["spec"]

    nodes: List[Dict[str, Any]] = []  # [{name, role, device}, ...]
    edges: List[Dict[str, Any]] = (
        []
    )  # [{u, v, edge_type, edge_role, V_diff, C, gds, R}, ...]
    node_idx: Dict[Tuple[str, str], int] = {}  # (device.pin or net, role) -> idx
    net_node_idx: Dict[str, int] = {}

    def add_node(name: str, role: str, device: Optional[str] = None) -> int:
        """
        add a node to the nodes list, return the index of the node
        """
        idx = len(nodes)
        nodes.append({"name": name, "role": role, "device": device})
        return idx

    # 1) first create all net nodes (include vdda/gnda)
    for net in all_nets:
        s = norm_name(net)
        if s in supplies_map:
            role = supplies_map[s]  # VDD/GND
            idx = add_node(role, role, device=None)
            net_node_idx[net] = idx

    # 2) for each MOS, create pin nodes + dummy, connect intra; then build small signal inter edges
    # mos_list example: MosLine(name='M1', d='net5', g='net1', s='vdda', b='vdda', model='sky130_fd_pr__pfet_01v8')
    # collection all net connected to each pin
    net_to_pins = {}
    for m in mos_list:
        # "M1"
        inst = m.name
        # pin nodes
        g = add_node(f"{inst}.G", "G", device=inst)
        d = add_node(f"{inst}.D", "D", device=inst)
        s = add_node(f"{inst}.S", "S", device=inst)
        b = add_node(f"{inst}.B", "B", device=inst)
        dummy = add_node(f"{inst}.dummy", "dummy", device=inst)

        # mapping the index (optional): (M1, G) -> g
        node_idx[(inst, "G")] = g
        node_idx[(inst, "D")] = d
        node_idx[(inst, "S")] = s
        node_idx[(inst, "B")] = b
        node_idx[(inst, "dummy")] = dummy

        # intra（pin↔dummy）
        for ni in [g, d, s, b]:
            edges.append(
                {
                    "u": dummy,
                    "v": ni,
                    "edge_type": EDGE_TYPE["intra"],
                    "edge_role": EDGE_ROLE["none"],
                    "V_diff": 0.0,
                    "C": 0.0,
                    "gds": 0.0,
                    "R": 0.0,
                    "current": 0.0,
                }
            )

        # intra：small signal edge  —— DAG
        def get(opname: str) -> float:
            return float(ops.get(inst.upper(), {}).get(opname, 0.0))

        # GS：Vgs, Cgs, role=gm_control（direction: G→S）
        edges.append(
            {
                "u": g,
                "v": s,
                "edge_type": EDGE_TYPE["intra"],
                "edge_role": EDGE_ROLE["gm_control"],
                "V_diff": get("vgs"),
                "C": get("cgs"),
                "gds": 0.0,
                "R": 0.0,
                "current": 0.0,
            }
        )

        # DS：Vds, gds, role=gm_target（direction: D→S）
        edges.append(
            {
                "u": d,
                "v": s,
                "edge_type": EDGE_TYPE["intra"],
                "edge_role": EDGE_ROLE["gm_target"],
                "V_diff": get("vds"),
                "C": 0.0,
                "gds": get("gds"),
                "R": 0.0,
                "current": 0.0,
            }
        )

        # GD：Cgd（no voltage difference）, direction: G→D
        edges.append(
            {
                "u": g,
                "v": d,
                "edge_type": EDGE_TYPE["intra"],
                "edge_role": EDGE_ROLE["none"],
                "V_diff": 0.0,
                "C": get("cgd"),
                "gds": 0.0,
                "R": 0.0,
                "current": 0.0,
            }
        )

        # DB/SB：Cdb / Csb direction: D→B, S→B
        edges.append(
            {
                "u": d,
                "v": b,
                "edge_type": EDGE_TYPE["intra"],
                "edge_role": EDGE_ROLE["none"],
                "V_diff": 0.0,
                "C": get("cdb"),
                "gds": 0.0,
                "R": 0.0,
                "current": 0.0,
            }
        )
        edges.append(
            {
                "u": s,
                "v": b,
                "edge_type": EDGE_TYPE["intra"],
                "edge_role": EDGE_ROLE["none"],
                "V_diff": -get("vbs"),
                "C": get("csb"),
                "gds": 0.0,
                "R": 0.0,
                "current": 0.0,
            }
        )

        # dummy node device-level scalar (as additional attribute)
        size_dict = sizes.get(inst.upper(), {})
        nodes[dummy]["dummy_feat"] = {
            "W/L": size_dict.get("wl_ratio", 0.0),
            "L": size_dict.get("l", 0.0),
            "M": size_dict.get("m", 0.0),
            "Id": get("id"),
            "gm": get("gm"),
            "gmbs": get("gmbs"),
            "vth": get("vth"),
            "vdsat": get("vdsat"),
            "mos_type": mos_type_id(m.model),
            # add corner/temp/VDD etc. for graph-level; leave empty here
        }

        # pin↔net inter connection (only connect gnd or vdd)
        for pin_name, ni in [(m.g, g), (m.d, d), (m.s, s), (m.b, b)]:
            # u is the source node, v is the target node
            if pin_name in net_node_idx:
                edges.append(
                    {
                        "u": ni,
                        "v": net_node_idx[pin_name],
                        "edge_type": EDGE_TYPE["inter"],
                        "edge_role": EDGE_ROLE["none"],
                        "V_diff": 0.0,
                        "C": 0.0,
                        "gds": 0.0,
                        "R": 0.0,
                        "current": 0.0,
                    }
                )

        # pin↔net inter connection (topology)
        # ni stands for the node index
        for pin_name, ni in [(m.g, g), (m.d, d), (m.s, s), (m.b, b)]:
            # not gnd or vdd
            if pin_name not in net_node_idx:
                if pin_name not in net_to_pins:
                    net_to_pins[pin_name] = []
                net_to_pins[pin_name].append(ni)

    # 3) capacitor: CT/CB node + one inter edge (edge_attr: C)
    # cap_list example: CapLine(name='C1', t='net1', b='vdda', model='sky130_fd_pr__cap_mim_m3_1')
    for c in cap_list:
        ct = add_node(f"{c.name}.T", "CT", device=c.name)
        cb = add_node(f"{c.name}.B", "CB", device=c.name)
        # only connect gnd or vdd
        if c.t in net_node_idx:
            edges.append(
                {
                    "u": ct,
                    "v": net_node_idx[c.t],
                    "edge_type": EDGE_TYPE["inter"],
                    "edge_role": EDGE_ROLE["none"],
                    "V_diff": 0.0,
                    "C": 0.0,
                    "gds": 0.0,
                    "R": 0.0,
                    "current": 0.0,
                }
            )
        if c.b in net_node_idx:
            edges.append(
                {
                    "u": cb,
                    "v": net_node_idx[c.b],
                    "edge_type": EDGE_TYPE["inter"],
                    "edge_role": EDGE_ROLE["none"],
                    "V_diff": 0.0,
                    "C": 0.0,
                    "gds": 0.0,
                    "R": 0.0,
                    "current": 0.0,
                }
            )
        # device edge: CT↔CB (C)
        # map by name
        edges.append(
            {
                "u": ct,
                "v": cb,
                "edge_type": EDGE_TYPE["intra"],
                "edge_role": EDGE_ROLE["capacitor"],
                "V_diff": 0.0,
                "C": ops.get(c.name.upper(), {}).get("capacitance", 0.0),
                "gds": 0.0,
                "R": 0.0,
                "current": 0.0,
            }
        )

        # add net to pin for capacitor
        if c.t not in net_node_idx:
            if c.t not in net_to_pins:
                net_to_pins[c.t] = []
            net_to_pins[c.t].append(ct)

        if c.b not in net_node_idx:
            if c.b not in net_to_pins:
                net_to_pins[c.b] = []
            net_to_pins[c.b].append(cb)

    # 4) create inter edges for current source
    for i in ibias_list:
        it = add_node(f"{i.name}.T", "IT", device=i.name)
        ib = add_node(f"{i.name}.B", "IB", device=i.name)

        # only connect gnd or vdd (for current source, we only need to connect to gnd or vdd, not both sides)
        if i.pos in net_node_idx:
            edges.append(
                {
                    "u": it,
                    "v": net_node_idx[i.pos],
                    "edge_type": EDGE_TYPE["inter"],
                    "edge_role": EDGE_ROLE["none"],
                    "V_diff": 0.0,
                    "C": 0.0,
                    "gds": 0.0,
                    "R": 0.0,
                    "current": 0.0,
                }
            )
        if i.neg in net_node_idx:
            edges.append(
                {
                    "u": ib,
                    "v": net_node_idx[i.neg],
                    "edge_type": EDGE_TYPE["inter"],
                    "edge_role": EDGE_ROLE["none"],
                    "V_diff": 0.0,
                    "C": 0.0,
                    "gds": 0.0,
                    "R": 0.0,
                    "current": 0.0,
                }
            )

        # intra: IT↔IB (current source)
        current_val = float(ops.get("ibias0", {}).get("ibias", 0.0))
        edges.append(
            {
                "u": it,
                "v": ib,
                "edge_type": EDGE_TYPE["intra"],
                "edge_role": EDGE_ROLE["current_source"],
                "V_diff": 0.0,
                "C": 0.0,
                "gds": 0.0,
                "R": 0.0,
                "current": current_val,
            }
        )

        # add to normal net inter connection
        if i.pos not in net_node_idx:
            if i.pos not in net_to_pins:
                net_to_pins[i.pos] = []
            net_to_pins[i.pos].append(it)

        if i.neg not in net_node_idx:
            if i.neg not in net_to_pins:
                net_to_pins[i.neg] = []
            net_to_pins[i.neg].append(ib)

    # 5) create inter edges for normal nets
    for net, pins in net_to_pins.items():
        # only connect multiple pins
        if len(pins) > 1:
            for i in range(len(pins)):
                for j in range(i + 1, len(pins)):
                    edges.append(
                        {
                            "u": pins[i],
                            "v": pins[j],
                            "edge_type": EDGE_TYPE["inter"],
                            "edge_role": EDGE_ROLE["none"],
                            "V_diff": 0.0,
                            "C": 0.0,
                            "gds": 0.0,
                            "R": 0.0,
                            "current": 0.0,
                        }
                    )

    # 6) pack (use general dict; you can also change to PyG Data)
    graph = {
        "nodes": nodes,
        "edges": edges,
        "meta": {
            "pin_role_map": PIN_ROLE,
            "edge_type_map": EDGE_TYPE,
            "edge_role_map": EDGE_ROLE,
        },
    }
    return graph

### Test Script


In [ ]:
from pathlib import Path
import pandas as pd

netlist_path = "./circuits/opamp/tsm/netlist.j2"
csv_path = "./output/opamp/tsm/tsm_batch_results.csv"
out_dir = "./output/opamp/tsm/graphs"

df = pd.read_csv(csv_path)

for row_idx in range(1):
    all_params = read_data_from_csv(csv_path, row_idx)

    print(f"============== Row {row_idx} ==============")
    print(f"First line Meta gain: {all_params['spec']['meta::dcgain_']}")
    print(f"First line Meta phase margin: {all_params['spec']['meta::phase_margin']}")
    print(f"First line Meta GBW: {all_params['spec']['meta::gain_bandwidth_product_']}")
    print(f"First line Meta power: {all_params['spec']['meta::power']}")
    print(f"First line M1 size: {all_params['size']['M1']}")
    print(f"First line M1 op: {all_params['op']['M1']}")
    print(f"First line C1 size: {all_params['size']['C1']}")
    print(f"First line C1 op: {all_params['op']['C1']}")
    print(f"First line Ibias size: {all_params['op']['ibias0']}")

netlist_str = Path(netlist_path).read_text()
mos_list, cap_list, ibias_list, supplies_map, all_nets = parse_netlist_lines(
    netlist_str
)
print(f"============== Netlist lines parsed ==============")
print(f"mos_list: {mos_list}")
print(f"cap_list: {cap_list}")
print(f"supplies_map: {supplies_map}")
print(f"all_nets: {all_nets}")

# test build graph
print(f"============== Building Graph ==============")
graph = build_graph_from_netlist_and_csv(netlist_str, csv_path, 0)
print(f"Graph nodes count: {len(graph['nodes'])}")
print(f"Graph edges count: {len(graph['edges'])}")
print(f"Graph meta: {graph['meta']}")


def print_graph_info(graph):
    print(f"Graph Statistics:")
    print(f"  Nodes: {len(graph['nodes'])}")
    print(f"  Edges: {len(graph['edges'])}")
    print(f"  Edge types: {graph['meta']['edge_type_map']}")
    print(f"  Edge roles: {graph['meta']['edge_role_map']}")

    print(f"\nNodes:")
    for i, node in enumerate(graph["nodes"]):
        print(f"  {i}: {node}")

    print(f"\nEdges:")
    for i, edge in enumerate(graph["edges"]):
        print(f"  {i}: {edge}")


# print graph via text format
print(f"============== Graph Details ==============")
print_graph_info(graph)

============== Row 0 ==============
First line Meta gain: 87.05569
First line Meta phase margin: 0.0241353
First line Meta GBW: 7803112.0
First line Meta power: 335.7527
First line M1 size: {'l': 0.74, 'm': 75.0, 'w': 2.89, 'wl_ratio': 3.9}
First line M1 op: {'cdb': -4.7252e-14, 'cgd': 2.186229e-16, 'cgs': -5.04185e-13, 'csb': -7.0805e-14, 'gds': 1.072116e-06, 'gm': 0.0003418223, 'gmbs': 8.477349e-05, 'id': 1.969444e-05, 'vbs': 1.200906e-11, 'vds': 0.9485322, 'vdsat': 0.06416054, 'vgs': 0.9485322, 'vth': 0.9897955}
First line C1 size: {'l': 88.0, 'w': 52.0}
First line C1 op: {'capacitance': 9.198182e-12}
First line Ibias size: {'ibias': 9.8e-05}
============== Netlist lines parsed ==============
mos_list: [MosLine(name='M2', d='net5', g='net1', s='vdda', b='vdda', model='sky130_fd_pr__pfet_01v8'), MosLine(name='M3', d='net1', g='vinn', s='net2', b='gnda', model='sky130_fd_pr__nfet_01v8'), MosLine(name='M1', d='net1', g='net1', s='vdda', b='vdda', model='sky130_fd_pr__pfet_01v8'), MosLi

## GNN RL Graph


### Graph Builder


In [ ]:
# graph_builder.py  ——  Parse netlist (Sky130) + Params from outside -> Graph (PyG Data or dict)
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional, Any
import re
import torch
import numpy as np
from collections import defaultdict

# ======== Schema enums ========
PIN_ROLE = {
    "G": 0,
    "D": 1,
    "S": 2,
    "B": 3,  # MOS pins
    "RT": 4,
    "RB": 5,  # Resistor pins
    "CT": 6,
    "CB": 7,  # Capacitor pins
    "IT": 8,
    "IB": 9,  # Current source
    "VDD": 10,
    "GND": 11,  # Supplies
    "NET": 13,
    "VINN": 14,
    "VINP": 15,
    "VOUT": 16,
}


# ======== small tool ========
def inst_key(raw: str) -> str:
    # "XM2" / "M2" / "m2" -> "M2"
    m = re.search(r"([xX]?[mM]\d+)", raw)
    return m.group(1).upper().replace("XM", "M") if m else raw.upper()


def is_pfet(model: str) -> bool:
    return "pfet" in model.lower()


def mos_type_id(model: str) -> int:
    """0 for nfet, 1 for pfet"""
    return 1 if is_pfet(model) else 0


def norm_name(n: str) -> str:
    return n.strip().lower()


# ======== analyze netlist lines (Sky130 MOS: D G S B)=======
@dataclass
class MosLine:
    name: str  # "M1"
    d: str
    g: str
    s: str
    b: str
    model: str  # "sky130_fd_pr__pfet_01v8"


@dataclass
class CapLine:
    name: str  # "C1"
    t: str
    b: str
    model: str  # "sky130_fd_pr__cap_mim_m3_1"


# TODO: support multi current source
@dataclass
class IbiasLine:
    name: str  # "I0"
    pos: str  # "vdda"
    neg: str  # "net3"


def parse_netlist_lines(
    text: str,
) -> Tuple[List[MosLine], List[CapLine], List[IbiasLine], Dict[str, str], List[str]]:
    """
    Parse the netlist lines and return the list of MosLine, CapLine, IbiasLine, interface map and all net names
    """
    mos_list: List[MosLine] = []
    cap_list: List[CapLine] = []
    ibias_list: List[IbiasLine] = []
    # interface map, extendable
    interface_map = {
        "vdda": "VDD",
        "gnda": "GND",
        "vinn": "VINN",
        "vinp": "VINP",
        "vout": "VOUT",
    }
    all_nets: set[str] = set()

    for raw in text.splitlines():
        s = raw.strip()
        # remove comments, subcircuit interface and empty lines
        if not s or s.startswith(("*", ";", "//", ".")):
            continue

        # MOS：  XM2 D G S B model
        if re.match(r"^[xX]?[mM]\d+\s", s):
            toks = s.split()
            name = inst_key(toks[0])  # XM2 -> M2
            d, g, s_, b = toks[1:5]  # Sky130 mosfet pin order: D G S B
            model = toks[5]

            mos_list.append(MosLine(name=name, d=d, g=g, s=s_, b=b, model=model))
            all_nets.update([d, g, s_, b])
            continue

        # CAP： XC1 nodeT nodeB model
        if re.match(r"^[xX]?[cC]\d+\s", s):
            toks = s.split()
            name = toks[0].upper().replace("XC", "C").replace("c", "C")
            t, b = toks[1], toks[2]
            model = toks[3]

            cap_list.append(CapLine(name=name, t=t, b=b, model=model))
            all_nets.update([t, b])
            continue

        # TODO: support multi current source, need to modify the regex
        # current source: I0 pos neg
        if re.match(r"^I0\s", s):
            toks = s.split()
            name = toks[0].upper()
            pos, neg = toks[1], toks[2]

            ibias_list.append(IbiasLine(name=name, pos=pos, neg=neg))
            all_nets.update([pos, neg])
            continue

        # other: only collect net names for later use
        toks = s.split()
        if len(toks) >= 3:
            all_nets.update(toks[1:3])

    # record all net names (for creating NET nodes)
    return mos_list, cap_list, ibias_list, interface_map, sorted(all_nets)


# ==== helper: symmetric minmax to [-1,1] ====
def norm_minmax_sym(val: float, vmin: float, vmax: float) -> float:
    if vmax == vmin:
        return 0.0
    return 2.0 * ((val - vmin) / (vmax - vmin)) - 1.0


# def build_device_net_graph(
#     netlist_text: str,
#     sizing_dict: Dict[str, float],
#     group_table: Dict[str, int],
#     param_ranges: Dict[str, Tuple[float, float]],
#     max_groups: int = 5,
#     mode: str = "homo",
#     # mode: "homo" -> remove duplicates (only keep device<->net connection)
#     #       "hetero" -> keep duplicates (vdda/gnda/vinp/vinn/vout are all separate edges)
# ) -> Dict[str, Any]:
#     """
#     build a Device+Net graph, for GNN+RL.

#     nodes:
#         - Device nodes: M1/M2/.../C1/.../I0 physical devices
#         - Net nodes: net1/net2/.../vdda/gnda/vinp/vinn/vout signal lines
#         (includes Special Nets = VDD/GND/VINP/VINN/VOUT)

#     edges:
#         - Device <-> Net
#         - if mode="hetero": connect each pin once, no duplicates
#         - if mode="homo": only keep connection, remove duplicates

#     return:
#         {
#             "x": FloatTensor [N, D_feat],          # node features
#             "edge_index": LongTensor [2, E],       # directed (device->net, net->device)
#             "node_names": List[str],               # node names: "M1", "net3", ...
#             "is_device": FloatTensor [N],          # 1=device,0=net
#             "tunable_mask": FloatTensor [N],       # 1=tunable,0=not tunable
#             "group_ids": LongTensor [N],           # device group id; net is -1
#             "meta": { "high_level_classes": [...],
#                     "device_types": [...] }
#         }
#     """

#     # === parse netlist ===
#     mos_list, cap_list, ibias_list, interface_map, all_nets = parse_netlist_lines(
#         netlist_text
#     )

#     # === build device nodes ===
#     # record: name, type(=device type), class(=high level class), nets(=it connects to)
#     # we treat MOS as "active", CAP/IBIAS as "passive"
#     devices: List[Dict[str, Any]] = []

#     for m in mos_list:
#         type = "pmos" if is_pfet(m.model) else "nmos"
#         devices.append(
#             {
#                 "name": m.name,  # "M2"
#                 "type": type,  # "pmos"/"nmos"
#                 "class": "active",  # high level class
#                 "nets": [m.d, m.g, m.s, m.b],  # include all pins; duplicates allowed
#             }
#         )

#     for c in cap_list:
#         devices.append(
#             {
#                 "name": c.name,  # "C1"
#                 "type": "cap",
#                 "class": "passive",
#                 "nets": [c.t, c.b],
#             }
#         )

#     for ib in ibias_list:
#         devices.append(
#             {
#                 "name": ib.name,  # "I0"
#                 "type": "ibias",
#                 "class": "passive",
#                 "nets": [ib.pos, ib.neg],
#             }
#         )

#     # TODO: if you add resistors later:
#     # devices.append({"name": R1, "type": "res", "class": "passive", "nets":[n1,n2]})

#     # number of device nodes
#     Nd = len(devices)

#     # === build net nodes ===
#     # we treat each net (all_nets) as a node
#     # - if it's in interface_map (vdda -> "VDD"), we treat it as special net
#     # - otherwise it's generic internal net
#     nets: List[Dict[str, Any]] = []
#     for net in all_nets:
#         # lowercase
#         norm_net = norm_name(net)
#         if norm_net in interface_map:
#             # e.g. "VDD","VINN",...
#             mapped = interface_map[norm_net]
#             nets.append(
#                 {
#                     "name": net,  # original net name "vdda"
#                     "type": mapped.lower(),  # "vdd","vinn","vinp","vout","gnd"
#                     "class": "net_special",  # high_level_class for special
#                 }
#             )
#         else:
#             nets.append(
#                 {
#                     "name": net,  # "net1"
#                     "type": "net",  # generic net
#                     "class": "net_generic",
#                 }
#             )

#     # number of net nodes
#     Nn = len(nets)
#     # total number of nodes
#     total_nodes = Nd + Nn

#     # assign global index to device / net nodes
#     dev_name_to_idx = {dev["name"]: i for i, dev in enumerate[Dict[str, Any]](devices)}
#     net_name_to_idx = {netinfo["name"]: (Nd + j) for j, netinfo in enumerate[Dict[str, Any]](nets)}

#     # TODO: Stop here, @date: 2025-10-28
#     # === build edges Device<->Net ===
#     # hetero mode: no duplicates (keep multiple connections from same device to same net)
#     # homo   mode: remove duplicates (only keep one connection per device/net pair)
#     edge_set = set[Any]()
#     edge_list = []

#     # for each device
#     for i_dev, dev in enumerate[Dict[str, Any]](devices):
#         # if mode==homo, first remove duplicates in nets; otherwise keep original order and duplicates
#         nets_for_this = dev["nets"] if mode == "hetero" else list[Any](set[Any](dev["nets"]))

#         for net_name in nets_for_this:
#             if net_name not in net_name_to_idx:
#                 # net not recognized? skip
#                 continue
#             # get index for the net node in the global net list index
#             j_net = net_name_to_idx[net_name]

#             if mode == "homo":
#                 # homo: we only need one device->net and one net->device edge
#                 if (i_dev, j_net) not in edge_set:
#                     #? here we add both directions of the edge since in PyG, edge is directed
#                     edge_set.add((i_dev, j_net))
#                     edge_set.add((j_net, i_dev))
#                     edge_list.append((i_dev, j_net))
#                     edge_list.append((j_net, i_dev))
#             else:
#                 # hetero: push every connection
#                 edge_list.append((i_dev, j_net))
#                 edge_list.append((j_net, i_dev))

#     if len(edge_list) == 0:
#         edge_index = torch.zeros((2, 0), dtype=torch.long)
#     else:
#         edge_index = torch.tensor(edge_list, dtype=torch.long).t()  # [2, E]

#     # === 5. define vocabularies for node features ===
#     # high_level_class one-hot:
#     #   - "active"       (MOS)
#     #   - "passive"      (cap, ibias, res)
#     #   - "net_special"  (VINP/VINN/VOUT/VDD/GND)
#     #   - "net_generic"  (internal nets: net1/net2/...)
#     #   - "other"        (fallback)
#     high_level_classes = [
#         "active",
#         "passive",
#         "net_special",
#         "net_generic",
#         "other",
#     ]

#     # device_type one-hot:
#     # includes both device kinds and special net kinds
#     device_types = [
#         "nmos",
#         "pmos",
#         "res",
#         "cap",
#         "ibias",
#         "vinp",
#         "vinn",
#         "vout",
#         "vdd",
#         "gnd",
#         "net",
#     ]
#     # generic internal nets -> "net"
#     # special nets use their own: "vinp","vinn","vout","vdd","gnd"

#     def one_hot(idx: int, size: int) -> np.ndarray:
#         v = np.zeros(size, dtype=np.float32)
#         v[idx] = 1.0
#         return v

#     def class_onehot(cls_name: str) -> np.ndarray:
#         if cls_name in high_level_classes:
#             k = high_level_classes.index(cls_name)
#         else:
#             k = high_level_classes.index("other")
#         return one_hot(k, len(high_level_classes))  # [5]

#     def type_onehot(kind: str) -> np.ndarray:
#         # kind can be: "nmos","pmos","cap","ibias","res",
#         #              "vinp","vinn","vout","vdd","gnd","net"
#         t = kind
#         # normalize generic net label
#         if t not in device_types:
#             if t in ["net_generic", "net_special"]:
#                 t = "net"
#             elif t == "net":
#                 pass
#             else:
#                 # fallback, treat as "net"
#                 t = "net"
#         k = device_types.index(t)
#         return one_hot(k, len(device_types))  # [len(device_types)]

#     def group_onehot_for_device(dev_name: str) -> np.ndarray:
#         gid = group_table.get(dev_name, max_groups - 1)
#         if gid >= max_groups:
#             gid = max_groups - 1
#         v = np.zeros(max_groups, dtype=np.float32)
#         v[gid] = 1.0
#         return v

#     def empty_group_onehot() -> np.ndarray:
#         return np.zeros(max_groups, dtype=np.float32)

#     def is_tunable(kind: str, cls_name: str) -> float:
#         # 我们假设:
#         # - 物理器件 (nmos/pmos/cap/ibias/res) = 可调 1
#         # - net (vdd/gnd/vinp/...) = 不可调 0
#         if cls_name.startswith("net"):
#             return 0.0
#         return 1.0 if kind in ["nmos", "pmos", "cap", "ibias", "res"] else 0.0

#     def params_vec_for_node(node_name: str, kind: str, cls_name: str) -> np.ndarray:
#         # 对 device: [W, L, M, R, C, I] 归一化
#         # 对 net: zeros
#         if cls_name.startswith("net"):
#             return np.zeros(6, dtype=np.float32)

#         # 拉 sizing_dict:
#         # 你说了是 {name}_l 这种形式；我下面用大写键风格，
#         # 你可以按实际命名改:
#         #   M1_W, M1_L, M1_M
#         #   C1_C
#         #   I0_I
#         #   (R1_R 如果以后有电阻)
#         W_raw = sizing_dict.get(f"{node_name}_W", 0.0)
#         L_raw = sizing_dict.get(f"{node_name}_L", 0.0)
#         M_raw = sizing_dict.get(f"{node_name}_M", 0.0)
#         R_raw = sizing_dict.get(f"{node_name}_R", 0.0)
#         C_raw = sizing_dict.get(f"{node_name}_C", 0.0)
#         I_raw = sizing_dict.get(f"{node_name}_I", 0.0)

#         def nm(x, key):
#             lo, hi = param_ranges[key]
#             return norm_minmax_sym(x, lo, hi) if hi > lo else 0.0

#         return np.array(
#             [
#                 nm(W_raw, "W"),
#                 nm(L_raw, "L"),
#                 nm(M_raw, "M"),
#                 nm(R_raw, "R"),
#                 nm(C_raw, "C"),
#                 nm(I_raw, "I"),
#             ],
#             dtype=np.float32,
#         )

#     # === 6. build node features for ALL nodes (devices first, then nets)
#     x_rows: List[np.ndarray] = []
#     tunable_mask_list: List[float] = []
#     group_id_list: List[int] = []
#     node_names: List[str] = []
#     is_device_list: List[float] = []

#     # 6a. devices
#     for dev in devices:
#         dev_name = dev["name"]  # "M2"
#         kind = dev["kind"]  # "pmos","nmos","cap","ibias"...
#         cls_name = dev["class"]  # "active","passive"

#         cls_oh = class_onehot(cls_name)  # e.g. active -> onehot[5]
#         type_oh = type_onehot(kind)  # e.g. pmos  -> onehot[11]
#         group_oh = group_onehot_for_device(dev_name)  # [max_groups]
#         t_mask = np.array([is_tunable(kind, cls_name)], dtype=np.float32)  # [1]
#         params_v = params_vec_for_node(dev_name, kind, cls_name)  # [6]

#         feat = np.concatenate([cls_oh, type_oh, group_oh, t_mask, params_v], axis=0)

#         x_rows.append(feat)
#         tunable_mask_list.append(t_mask[0])
#         group_id_list.append(group_table.get(dev_name, max_groups - 1))
#         node_names.append(dev_name)
#         is_device_list.append(1.0)

#     # 6b. nets
#     for netinfo in nets:
#         net_name = netinfo["name"]  # "gnda","net1","vdda","vinn",...
#         kind = netinfo["kind"]  # "gnd","net","vdd","vinn","vinp",...
#         cls_name = netinfo["class"]  # "net_special" or "net_generic"

#         cls_oh = class_onehot(cls_name)
#         type_oh = type_onehot(kind)
#         group_oh = np.zeros(max_groups, dtype=np.float32)  # nets 没有group
#         t_mask = np.array([is_tunable(kind, cls_name)], dtype=np.float32)  # expect 0
#         params_v = np.zeros(6, dtype=np.float32)  # nets 没参数

#         feat = np.concatenate([cls_oh, type_oh, group_oh, t_mask, params_v], axis=0)

#         x_rows.append(feat)
#         tunable_mask_list.append(t_mask[0])
#         group_id_list.append(-1)  # nets: group = -1
#         node_names.append(net_name)
#         is_device_list.append(0.0)

#     # stack to torch tensors
#     x = torch.tensor(np.stack(x_rows, axis=0), dtype=torch.float32)  # [N, D]
#     tunable_mask_tensor = torch.tensor(tunable_mask_list, dtype=torch.float32)  # [N]
#     group_ids_tensor = torch.tensor(group_id_list, dtype=torch.long)  # [N]
#     is_device_tensor = torch.tensor(is_device_list, dtype=torch.float32)  # [N]

#     # === 7. final pack ===
#     graph: Dict[str, Any] = {
#         "x": x,
#         "edge_index": edge_index,  # [2, E]
#         "node_names": node_names,  # [N]
#         "is_device": is_device_tensor,  # [N]
#         "tunable_mask": tunable_mask_tensor,  # [N]
#         "group_ids": group_ids_tensor,  # [N]
#         "meta": {
#             "high_level_classes": high_level_classes,
#             "device_types": device_types,
#             "mode": mode,
#         },
#     }

#     return graph


def build_device_net_graph(
    netlist_text: str,
    sizing_dict: Dict[str, float],
    group_table: Dict[str, int],
    param_ranges: Dict[str, Tuple[float, float]],
    max_groups: int = 5,
    mode: str = "homo",  # "homo" or "hetero"
) -> Dict[str, Any]:
    """

    generate bipartite graph:
        - Node = Devices + Nets
        - Edge = Device <-> Net

    new features:
        - hetero mode preserves pin-level connections
        - homo mode merges multiple pins into a single edge
        - returns edge_attr as pin_role one-hot

    parameters:
        netlist_text: str, netlist text
        sizing_dict: Dict[str, float], sizing dictionary
        group_table: Dict[str, int], group table
        param_ranges: Dict[str, Tuple[float, float]], parameter ranges
        max_groups: int, maximum number of groups, default=5
        mode: str, "homo" or "hetero", default="homo"

    returns:
        {
            "x": FloatTensor [N, D_node],
            "edge_index": LongTensor [2, E],
            "edge_attr": FloatTensor [E, D_edge],
            "node_names": List[str],
            "is_device": FloatTensor [N],
            "tunable_mask": FloatTensor [N],
            "group_ids": LongTensor [N],
            "meta": {...}
        }
    """

    mos_list, cap_list, ibias_list, interface_map, all_nets = parse_netlist_lines(
        netlist_text
    )

    # ---------------------------------------------------
    # pin role vocabulary for edge_attr
    #! note: MERGED is used in homo mode for merged edge
    PIN_ROLE_VOCAB = [
        "D",
        "G",
        "S",
        "B",  # MOS
        "CT",
        "CB",  # Capacitor terminals
        "IT",
        "IB",  # Current source terminals
        "MERGED",  # merged edge in homo mode
    ]
    pinrole_to_idx = {r: i for i, r in enumerate[str](PIN_ROLE_VOCAB)}

    # encode pin role, here we use one-hot encoding
    def pinrole_onehot(role: str) -> np.ndarray:
        idx = pinrole_to_idx[role]
        v = np.zeros(len(PIN_ROLE_VOCAB), dtype=np.float32)
        v[idx] = 1.0
        return v

    # ---------------------------------------------------
    # build device list with pin info
    #! we save pin, net pairs:
    # "pins": [("G","net1"), ("D","net5"), ...]
    devices: List[Dict[str, Any]] = []

    for m in mos_list:
        type = "pmos" if is_pfet(m.model) else "nmos"
        devices.append(
            {
                "name": m.name,  # "M2"
                "type": type,  # "pmos"/"nmos"
                "class": "active",
                "pins": [
                    ("D", m.d),
                    ("G", m.g),
                    ("S", m.s),
                    ("B", m.b),
                ],
            }
        )

    for c in cap_list:
        devices.append(
            {
                "name": c.name,  # "C1"
                "type": "cap",
                "class": "passive",
                "pins": [
                    ("CT", c.t),
                    ("CB", c.b),
                ],
            }
        )

    for ib in ibias_list:
        devices.append(
            {
                "name": ib.name,  # "I0"
                "type": "ibias",
                "class": "passive",
                "pins": [
                    ("IT", ib.pos),
                    ("IB", ib.neg),
                ],
            }
        )

    # number of devices
    Nd = len(devices)

    # ---------------------------------------------------
    # build net nodes
    nets: List[Dict[str, Any]] = []
    for net in all_nets:
        norm_net = norm_name(net)
        if norm_net in interface_map:
            mapped = interface_map[norm_net]  # e.g. "VDD","VINN","VOUT",...
            nets.append(
                {
                    "name": net,
                    "type": mapped.lower(),  # "vdd","vinn","vinp","vout","gnd"
                    "class": "net_special",
                }
            )
        else:
            nets.append(
                {
                    "name": net,
                    "type": "net",  # generic net
                    "class": "net_generic",
                }
            )

    # number of nets
    Nn = len(nets)
    # total number of nodes
    total_nodes = Nd + Nn

    # index maps
    dev_name_to_idx = {dev["name"]: i for i, dev in enumerate[Dict[str, Any]](devices)}
    net_name_to_idx = {n["name"]: (Nd + j) for j, n in enumerate[Dict[str, Any]](nets)}

    # ---------------------------------------------------
    # build edges and edge_attr
    edge_list: List[Tuple[int, int]] = []
    edge_attr_list: List[np.ndarray] = []

    if mode == "hetero":
        # (a) hetero: each pin has a separate edge, keep pin role
        for i_dev, dev in enumerate[Dict[str, Any]](devices):
            for role, net_name in dev["pins"]:
                if net_name not in net_name_to_idx:
                    continue
                # get index of the pin connected to this device
                j_net = net_name_to_idx[net_name]

                # encode pin role
                pin_role_one_hot = pinrole_onehot(role)

                # device -> net
                edge_list.append((i_dev, j_net))
                edge_attr_list.append(pin_role_one_hot)

                # net -> device
                edge_list.append((j_net, i_dev))
                edge_attr_list.append(pin_role_one_hot)

    else:
        # (b) homo: merge multiple pins of the same device to the same net
        # we first aggregate (device, net) pairs, collect their roles
        # for example, M1 has pins ["S","B"] connected to vdda => merge into one edge, role="MERGED"
        merged_edges = defaultdict[Any, set](set)  # (i_dev, j_net) -> set({"S","B"})
        for i_dev, dev in enumerate[Dict[str, Any]](devices):
            for role, net_name in dev["pins"]:
                if net_name not in net_name_to_idx:
                    continue
                j_net = net_name_to_idx[net_name]
                merged_edges[(i_dev, j_net)].add(role)

        for (i_dev, j_net), role_set in merged_edges.items():
            # only put one edge (bidirectional), edge attr is MERGED
            pin_role_one_hot = pinrole_onehot("MERGED")

            edge_list.append((i_dev, j_net))
            edge_attr_list.append(pin_role_one_hot)

            edge_list.append((j_net, i_dev))
            edge_attr_list.append(pin_role_one_hot)

    # build edge_index and edge_attr
    if len(edge_list) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, len(PIN_ROLE_VOCAB)), dtype=torch.float32)
    else:
        edge_index = torch.tensor(edge_list, dtype=torch.long).t()  # [2,E]
        edge_attr = torch.tensor(
            np.stack(edge_attr_list, axis=0), dtype=torch.float32
        )  # [E,D_edge]

    # ---------------------------------------------------
    # vocabularies for node features
    high_level_classes = [
        # active devices
        "active",  # NMOS / PMOS
        # passive devices
        "passive",  # cap / ibias / res
        # nets
        "net_special",  # supplies(VSS / GND ...) / IO (VINN / VINP / VOUT/ ...)
        "net_generic",  # internal nets (net1, net2, ...)
        # Undefined class will be classified as "other"
        "other",
    ]

    # ~ Node types, future extension can be added here
    device_types = [
        "nmos",
        "pmos",
        "res",
        "cap",
        "ibias",
        "vinp",
        "vinn",
        "vout",
        "vdd",
        "gnd",
        "net",
    ]

    def one_hot(idx: int, size: int) -> np.ndarray:
        v = np.zeros(size, dtype=np.float32)
        v[idx] = 1.0
        return v

    def class_onehot(cls_name: str) -> np.ndarray:
        if cls_name in high_level_classes:
            k = high_level_classes.index(cls_name)
        else:
            k = high_level_classes.index("other")
        return one_hot(k, len(high_level_classes))

    def type_onehot(kind: str) -> np.ndarray:
        t = kind
        if t not in device_types:
            # map fallbacks: net_generic/net_special -> "net"
            if t in ["net_special", "net_generic", "net"]:
                t = "net"
            else:
                t = "net"
        k = device_types.index(t)
        return one_hot(k, len(device_types))

    def group_onehot_for_device(dev_name: str) -> np.ndarray:
        gid = group_table.get(dev_name, max_groups - 1)
        if gid >= max_groups:
            gid = max_groups - 1
        v = np.zeros(max_groups, dtype=np.float32)
        v[gid] = 1.0
        return v

    def zero_group_onehot() -> np.ndarray:
        return np.zeros(max_groups, dtype=np.float32)

    def is_tunable(kind: str, cls_name: str) -> float:
        # device (nmos/pmos/cap/ibias/res): 1, which means tunable
        # nets: 0, which means non-tunable
        if cls_name.startswith("net"):
            return 0.0
        return 1.0 if kind in ["nmos", "pmos", "cap", "ibias", "res"] else 0.0

    def params_vec_for_node(node_name: str, type: str, cls_name: str) -> np.ndarray:
        if cls_name.startswith("net"):
            return np.zeros(6, dtype=np.float32)

        W_raw = sizing_dict.get(f"{node_name}_W", 0.0)
        L_raw = sizing_dict.get(f"{node_name}_L", 0.0)
        M_raw = sizing_dict.get(f"{node_name}_M", 0.0)
        R_raw = sizing_dict.get(f"{node_name}_R", 0.0)
        C_raw = sizing_dict.get(f"{node_name}_C", 0.0)
        I_raw = sizing_dict.get(f"{node_name}_I", 0.0)

        def nm(x, key):
            lo, hi = param_ranges[key]
            return norm_minmax_sym(x, lo, hi) if hi > lo else 0.0

        return np.array(
            [
                nm(W_raw, "W"),
                nm(L_raw, "L"),
                nm(M_raw, "M"),
                nm(R_raw, "R"),
                nm(C_raw, "C"),
                nm(I_raw, "I"),
            ],
            dtype=np.float32,
        )

    # ---------------------------------------------------
    # assemble node features
    x_rows: List[np.ndarray] = []
    tunable_mask_list: List[float] = []
    group_id_list: List[int] = []
    node_names: List[str] = []
    is_device_list: List[float] = []

    # devices
    for dev in devices:
        dev_name = dev["name"]
        type = dev["type"]
        cls_name = dev["class"]

        cls_oh = class_onehot(cls_name)
        type_oh = type_onehot(type)
        group_oh = group_onehot_for_device(dev_name)
        tunable_mask = np.array([is_tunable(type, cls_name)], dtype=np.float32)
        params_v = params_vec_for_node(dev_name, type, cls_name)

        feat = np.concatenate(
            [cls_oh, type_oh, group_oh, tunable_mask, params_v], axis=0
        )

        x_rows.append(feat)
        tunable_mask_list.append(tunable_mask[0])
        group_id_list.append(group_table.get(dev_name, max_groups - 1))
        node_names.append(dev_name)
        is_device_list.append(1.0)

    # nets
    for netinfo in nets:
        net_name = netinfo["name"]
        type = netinfo["type"]
        cls_name = netinfo["class"]

        cls_oh = class_onehot(cls_name)
        type_oh = type_onehot(type)
        group_oh = zero_group_onehot()
        tunable_mask = np.array(
            [is_tunable(type, cls_name)], dtype=np.float32
        )  # expect 0
        params_v = np.zeros(6, dtype=np.float32)

        feat = np.concatenate(
            [cls_oh, type_oh, group_oh, tunable_mask, params_v], axis=0
        )

        x_rows.append(feat)
        tunable_mask_list.append(tunable_mask[0])
        group_id_list.append(-1)
        node_names.append(net_name)
        is_device_list.append(0.0)

    x = torch.tensor(np.stack(x_rows, axis=0), dtype=torch.float32)
    # edge_index = edge_index  # already torch
    # edge_attr = edge_attr  # already torch
    tunable_mask_t = torch.tensor(tunable_mask_list, dtype=torch.float32)
    group_ids_t = torch.tensor(group_id_list, dtype=torch.long)
    is_device_t = torch.tensor(is_device_list, dtype=torch.float32)

    graph: Dict[str, Any] = {
        "x": x,
        "edge_index": edge_index,
        "edge_attr": edge_attr,  # <--- New!
        "node_names": node_names,
        "is_device": is_device_t,
        "tunable_mask": tunable_mask_t,
        "group_ids": group_ids_t,
        "meta": {
            "high_level_classes": high_level_classes,
            "device_types": device_types,
            "pin_role_vocab": PIN_ROLE_VOCAB,
            "mode": mode,
        },
    }

    return graph

### Test Script


In [ ]:
from pathlib import Path
import torch
import numpy as np
from pprint import pprint

### test netlist parser
netlist_path = "./circuits/opamp/tsm/netlist.j2"
netlist_str = Path(netlist_path).read_text()

mos_list, cap_list, ibias_list, interface_map, all_nets = parse_netlist_lines(
    netlist_str
)

print("============== Netlist lines parsed ==============")
print("mos_list:", mos_list)
print("cap_list:", cap_list)
print("ibias_list:", ibias_list)
print("interface_map:", interface_map)
print("all_nets:", all_nets)
print()

### test graph builder by fake sizing_dict

sizing_dict = {
    "M2_W": 5.46,
    "M2_L": 1.27,
    "M2_M": 58,
    "M3_W": 4.22,
    "M3_L": 1.03,
    "M3_M": 52,
    "M1_W": 5.46,
    "M1_L": 1.27,
    "M1_M": 58,
    "M4_W": 4.22,
    "M4_L": 1.03,
    "M4_M": 52,
    "M6_W": 15.25,
    "M6_L": 2.63,
    "M6_M": 49,
    "M5_W": 6.55,
    "M5_L": 1.26,
    "M5_M": 91,
    "M8_W": 35.0,
    "M8_L": 5.0,
    "M8_M": 100,
    "M7_W": 30.86,
    "M7_L": 4.06,
    "M7_M": 1,
    # fake cap value
    "C1_C": 10,
    # fake Ibias value
    "I0_I": 1e-4,
}

### test group_table
# 设计目的：同一个 group 的 device 在 RL 里强绑定，比如差分对 M3/M4，尾电流 M6/I0，输出级 M7/M8...
# 这个完全由你定义策略；这里给一个合理示例：
group_table = {
    "M3": 0,  # diff pair left
    "M4": 0,  # diff pair right
    "M6": 1,  # tail
    "I0": 1,  # bias source tie to tail
    "M7": 2,  # output pmos
    "M8": 2,  # output nmos
    "M1": 3,  # active load / mirror
    "M2": 3,  # active load / mirror
    "M5": 4,  # maybe internal bias stack
    "C1": 4,  # compensation cap grouped for now
    # 没出现的device会在 builder 里自动 fallback 到 max_groups-1
}

### test param_ranges
# 这个是做 min-max 归一化用的 [-1,1]。
# 先给一个比较宽松的工程范围，后面你可以从设计规则 or sweep 数据统计更新它：
param_ranges = {
    "W": (0.1, 100.0),  # µm
    "L": (0.1, 10.0),  # µm
    "M": (1.0, 200.0),  # multiplier
    "R": (1e3, 1e6),  # placeholder, 先没有R也可以放着
    "C": (1.0, 200),  # 我们上面用了 93*69=6417，所以范围给一个 O(1e4)
    "I": (1e-6, 5e-3),  # 偏置电流 1uA~5mA 粗范围
}

### 5. 调 build_device_net_graph (homo / hetero)

print("============== Build Graph: HOMO mode ==============")
graph_homo = build_device_net_graph(
    netlist_text=netlist_str,
    sizing_dict=sizing_dict,
    group_table=group_table,
    param_ranges=param_ranges,
    max_groups=5,
    mode="homo",
)

print("HOMO graph keys:", graph_homo.keys())
print("HOMO x.shape:", graph_homo["x"].shape)  # [N, D_node]
print("HOMO edge_index.shape:", graph_homo["edge_index"].shape)  # [2, E]
print("HOMO edge_attr.shape:", graph_homo["edge_attr"].shape)  # [E, D_edge]
print("HOMO num nodes:", len(graph_homo["node_names"]))
print("HOMO first 5 node_names:", graph_homo["node_names"][:5])
print("HOMO all node_names:", graph_homo["node_names"])
print("HOMO first 10 edges (src->dst):")
for e_idx in range(min(10, graph_homo["edge_index"].shape[1])):
    u = int(graph_homo["edge_index"][0, e_idx])
    v = int(graph_homo["edge_index"][1, e_idx])
    role_vec = graph_homo["edge_attr"][e_idx].tolist()
    print(
        f"  {e_idx}: {graph_homo['node_names'][u]} -> {graph_homo['node_names'][v]}, edge_attr(onehot)={role_vec}"
    )
print()

print("HOMO tunable_mask:", graph_homo["tunable_mask"])
print("HOMO group_ids:", graph_homo["group_ids"])
print("HOMO is_device:", graph_homo["is_device"])
print("HOMO meta:", graph_homo["meta"])
print()

print("============== Build Graph: HETERO mode ==============")
graph_hetero = build_device_net_graph(
    netlist_text=netlist_str,
    sizing_dict=sizing_dict,
    group_table=group_table,
    param_ranges=param_ranges,
    max_groups=5,
    mode="hetero",
)

print("HETERO x.shape:", graph_hetero["x"].shape)
print("HETERO edge_index.shape:", graph_hetero["edge_index"].shape)
print("HETERO edge_attr.shape:", graph_hetero["edge_attr"].shape)
print("HETERO num nodes:", len(graph_hetero["node_names"]))
print("HETERO first 10 edges (src->dst):")
for e_idx in range(min(10, graph_hetero["edge_index"].shape[1])):
    u = int(graph_hetero["edge_index"][0, e_idx])
    v = int(graph_hetero["edge_index"][1, e_idx])
    role_vec = graph_hetero["edge_attr"][e_idx].tolist()
    print(
        f"  {e_idx}: {graph_hetero['node_names'][u]} -> {graph_hetero['node_names'][v]}, edge_attr(onehot)={role_vec}"
    )
print()

print("HETERO meta:", graph_hetero["meta"])
print("pin_role_vocab:", graph_hetero["meta"]["pin_role_vocab"])

============== Netlist lines parsed ==============
mos_list: [MosLine(name='M2', d='net5', g='net1', s='vdda', b='vdda', model='sky130_fd_pr__pfet_01v8'), MosLine(name='M3', d='net1', g='vinn', s='net2', b='gnda', model='sky130_fd_pr__nfet_01v8'), MosLine(name='M1', d='net1', g='net1', s='vdda', b='vdda', model='sky130_fd_pr__pfet_01v8'), MosLine(name='M4', d='net5', g='vinp', s='net2', b='gnda', model='sky130_fd_pr__nfet_01v8'), MosLine(name='M6', d='net2', g='net3', s='gnda', b='gnda', model='sky130_fd_pr__nfet_01v8'), MosLine(name='M5', d='net3', g='net3', s='gnda', b='gnda', model='sky130_fd_pr__nfet_01v8'), MosLine(name='M8', d='vout', g='net3', s='gnda', b='gnda', model='sky130_fd_pr__nfet_01v8'), MosLine(name='M7', d='vout', g='net5', s='vdda', b='vdda', model='sky130_fd_pr__pfet_01v8')]
cap_list: [CapLine(name='C1', t='vout', b='net5', model='sky130_fd_pr__cap_mim_m3_1')]
ibias_list: [IbiasLine(name='I0', pos='vdda', neg='net3')]
interface_map: {'vdda': 'VDD', 'gnda': 'GND', 'v

### Visualize the graph


In [24]:
from pyvis.network import Network
import numpy as np
import json

ROLE_COLOR = {
    "device": "#1976D2",
    "net": "#9E9E9E",
    "vdd": "#F44336",
    "gnd": "#000000",
    "vinp": "#4CAF50",
    "vinn": "#8E24AA",
    "vout": "#FF9800",
}


def _pinrole_from_onehot(vec, vocab):
    idx = int(np.argmax(vec))
    return vocab[idx] if 0 <= idx < len(vocab) else "?"


def _decode_node_feature(vec, meta):
    """
    vec: 1D numpy array for a single node feature
    schema: [class_onehot(C), type_onehot(T), group_onehot(G), tunable(1), params(6)]
    """
    C = len(meta["high_level_classes"])
    T = len(meta["device_types"])
    # 自动反推 G
    D = vec.shape[0]
    G = D - (C + T + 1 + 6)
    if G < 0:
        raise ValueError(f"Feature dim not matching schema. D={D}, C={C}, T={T}")

    offs = 0
    cls_oh = vec[offs : offs + C]
    offs += C
    typ_oh = vec[offs : offs + T]
    offs += T
    grp_oh = vec[offs : offs + G]
    offs += G
    tun = vec[offs]
    offs += 1
    params = vec[offs : offs + 6]

    cls_idx = int(np.argmax(cls_oh)) if C > 0 else 0
    typ_idx = int(np.argmax(typ_oh)) if T > 0 else 0
    grp_idx = int(np.argmax(grp_oh)) if G > 0 else -1

    cls_name = meta["high_level_classes"][cls_idx] if C > 0 else "N/A"
    typ_name = meta["device_types"][typ_idx] if T > 0 else "N/A"

    return {
        "class": cls_name,
        "type": typ_name,
        "group": grp_idx,  # nets 通常为 -1（因为原始就是零向量）
        "tunable": float(tun),
        "params_norm": {
            "W": float(params[0]),
            "L": float(params[1]),
            "M": float(params[2]),
            "R": float(params[3]),
            "C": float(params[4]),
            "I": float(params[5]),
        },
        "G_dim": G,
    }


def show_pyvis_with_features(graph, height="800px", title="Circuit Graph", type="homo"):
    # ---- 取数据 ----
    names = graph["node_names"]
    is_device = graph["is_device"].cpu().numpy().tolist()
    xmat = graph["x"].cpu().numpy()  # [N, D]
    ei = graph["edge_index"].cpu().numpy()  # [2, E]
    ea = graph["edge_attr"].cpu().numpy()  # [E, D_edge]
    meta = graph["meta"]
    mode = meta.get("mode", "homo")
    pin_vocab = meta.get("pin_role_vocab", [])

    # ---- 初始化画布 ----
    net = Network(
        height=height,
        width="100%",
        directed=True,
        notebook=True,
        cdn_resources="in_line",
    )
    net.barnes_hut()

    # ---- 添加节点（含特征解码显示）----
    for i, name in enumerate(names):
        feat = _decode_node_feature(xmat[i], meta)

        # 颜色/形状
        if is_device[i] > 0.5:
            color, shape = ROLE_COLOR["device"], "dot"
            role_for_color = "device"
        else:
            low = name.lower()
            if low == "vdda":
                role_for_color = "vdd"
            elif low in ("gnda", "gnd"):
                role_for_color = "gnd"
            elif low == "vinp":
                role_for_color = "vinp"
            elif low == "vinn":
                role_for_color = "vinn"
            elif low == "vout":
                role_for_color = "vout"
            else:
                role_for_color = "net"
            color, shape = ROLE_COLOR[role_for_color], "box"

        # label：名称 + 简短摘要（类型/组/是否可调）
        tunable_flag = "✓" if feat["tunable"] > 0.5 else "✕"
        label = f"{name}\n{feat['type']} | G={feat['group']} | T={tunable_flag}"

        # tooltip：多行 HTML，显示完整信息和参数（归一化值）
        p = feat["params_norm"]
        tooltip = (
            f"<b>{name}</b><br>"
            f"class: {feat['class']}<br>"
            f"type: {feat['type']}<br>"
            f"group: {feat['group']}<br>"
            f"tunable: {tunable_flag}<br>"
            f"<hr style='margin:4px 0;'>"
            f"<b>Params (normalized [-1,1])</b><br>"
            f"W: {p['W']:.3f} &nbsp; L: {p['L']:.3f} &nbsp; M: {p['M']:.3f}<br>"
            f"R: {p['R']:.3f} &nbsp; C: {p['C']:.3f} &nbsp; I: {p['I']:.3f}"
        )

        net.add_node(i, label=label, title=tooltip, color=color, shape=shape)

    # ---- 添加边（显示 pin role）----
    for k in range(ei.shape[1]):
        u, v = int(ei[0, k]), int(ei[1, k])
        role = _pinrole_from_onehot(ea[k], pin_vocab)
        label = role if (mode == "hetero" or role == "MERGED") else ""
        net.add_edge(
            u, v, title=f"role={role}", label=label, color="#888888", arrows="to"
        )

    # ---- 可交互物理参数 ----
    options = {
        "physics": {
            "enabled": True,
            "barnesHut": {
                "gravitationalConstant": -20000,
                "centralGravity": 0.12,
                "springLength": 180,
                "springConstant": 0.02,
                "damping": 0.09,
                "avoidOverlap": 0.25,
            },
            "stabilization": {"iterations": 150},
        },
        "interaction": {
            "dragNodes": True,
            "dragView": True,
            "zoomView": True,
            "hover": True,
            "multiselect": True,
        },
        "nodes": {"font": {"size": 16}},
        "edges": {"smooth": {"type": "dynamic"}},
    }
    net.set_options(json.dumps(options))

    # 在 notebook 中内联显示；同时也会生成一个 preview.html 文件
    return net.show(f"preview_{type}.html")

show_pyvis_with_features(graph_homo, title="HOMO", type="homo")
show_pyvis_with_features(graph_hetero, title="HETERO", type="hetero")

preview_homo.html
preview_hetero.html
